# Agentic AI Tutorial: Building an Intelligent Agent

In [ ]:
!pip install "transformers>=5" accelerate ddgs requests

## Part 1: Setting Up the Language Model

We'll use Qwen2.5-0.5B-Instruct as our base language model. This model is:
- Instruction-tuned: Better at following specific instructions
- Relatively small (0.5B parameters): Suitable for educational purposes, fast even on CPU
- A chat model: It takes a list of messages (like the ChatGPT/Claude APIs) instead of raw text

> Note: older versions of this tutorial used FLAN-T5 with the `"text2text-generation"` pipeline. Transformers v5 removed that task — modern chat models with the `"text-generation"` pipeline replace it.

The model will serve as the "brain" of our agent, helping it decide which tools to use for different tasks.


In [ ]:
from transformers import pipeline

# Load a small public chat model
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="auto",        # half precision where supported -> smaller + faster
    device_map="auto",   # use the GPU automatically if one is available
)

def generate(prompt, max_new_tokens=20):
    """Send a single user message to the chat model and return its reply text."""
    messages = [{"role": "user", "content": prompt}]
    # do_sample=False = greedy decoding: deterministic answers, ideal for tool routing
    output = llm(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

## Part 2: Building Our First Tool - Calculator

Let's start by implementing a simple calculator tool. This will help us understand the basic structure of tool-using agents.

Key concepts:
1. Tool Definition: A function that performs a specific task
2. Tool Registry: A dictionary that maps tool names to their implementations
3. Prompt Engineering: Creating clear instructions for the model
4. Agent Loop: The main decision-making cycle

### Understanding the Code:
- The `calculator` function uses Python's `eval` to compute mathematical expressions
- The `tools` dictionary serves as our tool registry
- The `prompt_template` provides instructions to the language model
- The `agent_loop` manages the interaction between user, model, and tools


In [ ]:
# Tool 1: Calculator
def calculator(query):
    try:
        return str(eval(query))
    except Exception as e:
        return f"Error: {e}"

tools = {
    "calculator": calculator,
}

prompt_template = """You are a tool-routing agent. Pick the single best tool to answer the user's question. Do NOT answer the question yourself.

Available tools:
- calculator: use this for computing math expressions like 5*(3+2), 12/4, etc.

Instructions:
- Reply with only the name of the single best tool for this question, nothing else.

Question: {question}
Tool:"""

def agent_loop():
    while True:
        question = input("\nAsk me something: ")
        if question.lower() in ["exit", "quit"]:
            break

        prompt = prompt_template.format(question=question)
        thought = generate(prompt, max_new_tokens=10)
        print("Thought:", thought)

        if "calculator" in thought.lower():
            expr = ''.join(c for c in question if c.isdigit() or c in "+-*/.()")
            result = tools["calculator"](expr)
            print("Action: calculator")
            print("Input:", expr)
            print("Output:", result)
        else:
            print("Final Answer:", thought)

In [ ]:
agent_loop()

### Exercise 1: Understanding the Agent Loop

Try the agent with different mathematical expressions. Consider the following questions:

1. What happens if you input an invalid mathematical expression?
2. How does the agent decide to use the calculator tool?
3. What are the limitations of this simple calculator implementation?

Try these examples:
- 2*2
- 3+5*2
- sqrt(16) (Will this work? Why/why not?)


## Part 3: Adding Web Search Capabilities

Now we'll expand our agent's capabilities by adding a web search tool. This demonstrates how agents can interact with external services.

Key additions:
1. Web Search Tool: Using DuckDuckGo's search API
2. Multiple Tool Handling: How the agent chooses between tools


In [ ]:
from ddgs import DDGS

# Tool 2: Real web search via DuckDuckGo
def web_search(query):
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=1)
            for r in results:
                return f"{r['title']}: {r['body']}"
        return "No results found."
    except Exception as e:
        return f"Search error: {e}"

tools = {
    "calculator": calculator,
    "web_search": web_search,
}

prompt_template = """You are a tool-routing agent. Pick the single best tool to answer the user's question. Do NOT answer the question yourself.

Available tools:
- calculator: use this for computing math expressions like 5*(3+2), 12/4, etc.
- web_search: use this to look up facts, names, places, or current events

Examples:
Question: What is 12/4?
Tool: calculator
Question: Who won the 2022 World Cup?
Tool: web_search
Question: What is the capital of Japan?
Tool: web_search

Instructions:
- Reply with only the name of the single best tool for this question, nothing else.

Question: {question}
Tool:"""

def agent_loop():
    while True:
        question = input("\nAsk me something: ")
        if question.lower() in ["exit", "quit"]:
            break

        prompt = prompt_template.format(question=question)
        thought = generate(prompt, max_new_tokens=10)
        print("Thought:", thought)

        if "calculator" in thought.lower():
            expr = ''.join(c for c in question if c.isdigit() or c in "+-*/.()")
            result = tools["calculator"](expr)
            print("Action: calculator")
            print("Input:", expr)
            print("Output:", result)

        elif "web_search" in thought.lower():
            result = tools["web_search"](question)
            print("Action: web_search")
            print("Input:", question)
            print("Output:", result)

        else:
            print("Final Answer:", thought)


In [ ]:
agent_loop()

### Exercise 2: Testing Web Search Integration

Experiment with the enhanced agent by asking various questions:

1. Try factual queries (e.g., "Who is the prime minister of Canada?")
2. Try mathematical queries (How does the agent choose between calculator and web search?)
3. Try ambiguous queries (What happens?)

Questions to consider:
1. How reliable is the web search tool?
2. What are the limitations of using a single search result?
3. How could we improve the tool selection process?

## Part 4: Adding Weather Information

We'll now add a weather tool that demonstrates how to:
1. Make HTTP requests to external APIs
2. Handle real-time data
3. Process and format responses
4. Handle multiple tools with similar contexts

### Improving tool selection: constrained choices

With three tools, our small model starts making routing mistakes (e.g., sending "Is it raining in Montreal?" to `web_search`). A powerful trick for small models is to **constrain the output space**: instead of asking for a free-form tool name, we present the tools as a multiple-choice question and ask for a single letter. This is the same idea behind "structured outputs" in production agent frameworks — the less freedom the model has, the fewer ways it can go wrong.

In [ ]:
import requests

# Tool 3: retreive weather
def get_weather(city):
    try:
        # Format: wttr.in/<city>?format=3 gives a 1-line summary
        url = f"https://wttr.in/{city}?format=3"
        response = requests.get(url)
        if response.status_code == 200:
            return response.text.strip()
        return f"Error: Could not get weather for {city}"
    except Exception as e:
        return f"Weather lookup failed: {e}"

tools = {
    "calculator": calculator,
    "web_search": web_search,
    "get_weather": get_weather,
}

# With 3 tools we constrain the model to a multiple-choice answer:
# small models are far more reliable when they only have to pick a letter.
prompt_template = """Question: {question}

Which tool is needed to answer this question?
A) calculator - for math expressions like 5*(3+2), 12/4
B) web_search - for facts, names, places, or current events
C) get_weather - for current weather (temperature, rain, snow) in a city

Answer with a single letter (A, B, or C):"""

letter_to_tool = {"A": "calculator", "B": "web_search", "C": "get_weather"}


def agent_loop():
    while True:
        question = input("\nAsk me something: ")
        if question.lower() in ["exit", "quit"]:
            break

        prompt = prompt_template.format(question=question)
        thought = generate(prompt, max_new_tokens=5)
        tool_name = letter_to_tool.get(thought.strip().upper()[:1])
        print("Thought:", thought, "->", tool_name)

        if tool_name == "calculator":
            expr = ''.join(c for c in question if c.isdigit() or c in "+-*/.()")
            result = tools["calculator"](expr)
            print("Action: calculator")
            print("Input:", expr)
            print("Output:", result)

        elif tool_name == "web_search":
            result = tools["web_search"](question)
            print("Action: web_search")
            print("Input:", question)
            print("Output:", result)

        elif tool_name == "get_weather":
            # Extract city name (could improve)
            city = question.split()[-1]
            result = tools["get_weather"](city)
            print("Action: get_weather")
            print("Input:", city)
            print("Output:", result)

        else:
            print("Final Answer:", thought)

In [ ]:
agent_loop()